# Landslide4Sense — U-Net training

Trains the U-Net from [Rockfall_Prediction/training](https://github.com/Tiru1505/Rockfall_Prediction/tree/main/training) and exports it to ONNX.

**Before you start:** `Runtime -> Change runtime type -> T4 GPU`. On CPU this takes hours instead of ~40 minutes.

Run the cells in order. Total time is roughly 45–60 minutes, most of it cell 6.

## 1. Check the GPU

In [ ]:
import torch

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("NO GPU — go to Runtime > Change runtime type > T4 GPU, then rerun.")

## 2. Install dependencies and get the code

In [ ]:
!pip install -q h5py onnx onnxscript onnxruntime
!git clone -q https://github.com/Tiru1505/Rockfall_Prediction.git
%cd Rockfall_Prediction/training
!ls

## 3. Kaggle credentials

Get `kaggle.json` from <https://www.kaggle.com/settings> → **API** → *Create New Token*.
Run this cell and upload that file.

It is written to `~/.kaggle/` with `chmod 600`, which the Kaggle CLI requires. It lives only in this throwaway Colab VM — but it is a real credential, so don't commit it anywhere.

In [ ]:
from google.colab import files

print("Upload kaggle.json:")
files.upload()

!mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
!rm -f kaggle.json
print("credentials installed")

## 4. Download the dataset (~3.6 GB)

Two mirrors. The first matches the `TrainData/img` + `TrainData/mask` layout the loader expects; the second is the fallback.

You must be logged into Kaggle and have accepted the dataset's terms on its page, or the download returns 403.

In [ ]:
%%bash
set -e
mkdir -p /content/data && cd /content/data

if kaggle datasets download -d pypiahmad/landslide4sense-challenge 2>/dev/null; then
  echo "downloaded: pypiahmad/landslide4sense-challenge"
else
  echo "first mirror failed, trying the second..."
  kaggle datasets download -d tekbahadurkshetri/landslide4sense
fi

for z in *.zip; do unzip -q -o "$z" && rm -f "$z"; done
# Some mirrors ship a nested zip.
find . -name '*.zip' -exec sh -c 'unzip -q -o "$1" -d "$(dirname "$1")" && rm -f "$1"' _ {} \;
echo "done"

## 5. Locate and verify the data

Mirrors nest the folders differently, so this searches for the `img`/`mask` pair rather than assuming a path.

In [ ]:
from pathlib import Path

candidates = [
    p.parent for p in Path("/content/data").rglob("img")
    if p.is_dir() and (p.parent / "mask").is_dir() and any(p.glob("*.h5"))
]
assert candidates, (
    "No folder containing both img/ and mask/ with .h5 files was found under "
    "/content/data. Check the download cell's output."
)

# If several match, take the one with the most labelled patches (TrainData).
DATA_ROOT = max(candidates, key=lambda d: len(list((d / "mask").glob("*.h5"))))
n_img = len(list((DATA_ROOT / "img").glob("*.h5")))
n_mask = len(list((DATA_ROOT / "mask").glob("*.h5")))
print(f"DATA_ROOT = {DATA_ROOT}")
print(f"  images {n_img}  masks {n_mask}   (expect 3799 / 3799)")

# Confirm the shapes and the class imbalance this whole pipeline is built around.
import h5py, numpy as np
from dataset import list_pairs, positive_pixel_fraction

pairs = list_pairs(DATA_ROOT)
with h5py.File(pairs[0][0], "r") as f:
    print(f"  patch shape {f['img'].shape}   (expect (128, 128, 14))")

pos = positive_pixel_fraction(pairs)
print(f"  positive pixels: {pos*100:.2f}%  ->  all-background baseline would "
      f"score {100*(1-pos):.1f}% 'accuracy'")

## 6. Train (~40 min on a T4)

Watch **val IoU**, not loss — the checkpoint is saved on best validation IoU because loss can fall while the model gets worse at the only thing that matters.

In [ ]:
!python train.py --data-root "{DATA_ROOT}" --epochs 40 --batch-size 16 --workers 2

## 7. Evaluate on the held-out test split

**These are the numbers for your CV and README.** Report IoU and F1 — not accuracy.

In [ ]:
!python evaluate.py --data-root "{DATA_ROOT}" --checkpoint runs/unet/best.pt

## 8. Training curves

In [ ]:
import json
import matplotlib.pyplot as plt

history = json.load(open("runs/unet/history.json"))
epochs = [h["epoch"] for h in history]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(epochs, [h["train"]["loss"] for h in history], label="train")
ax1.plot(epochs, [h["val"]["loss"] for h in history], label="val")
ax1.set_xlabel("epoch"); ax1.set_ylabel("Dice+BCE loss"); ax1.set_title("Loss")
ax1.legend(); ax1.grid(alpha=0.3)

ax2.plot(epochs, [h["train"]["iou"] for h in history], label="train")
ax2.plot(epochs, [h["val"]["iou"] for h in history], label="val")
ax2.plot(epochs, [h["val"]["f1"] for h in history], label="val F1", ls="--")
ax2.set_xlabel("epoch"); ax2.set_ylabel("score"); ax2.set_title("IoU / F1")
ax2.legend(); ax2.grid(alpha=0.3)

plt.tight_layout()
plt.savefig("runs/unet/training_curves.png", dpi=150)
plt.show()
print(f"best val IoU: {max(h['val']['iou'] for h in history):.4f}")

## 9. Export to ONNX for the backend

In [ ]:
!python export_onnx.py --checkpoint runs/unet/best.pt --output landslide_unet.onnx

## 10. Download the results

Keep all of these — the ONNX model goes into the backend, and the metrics and curves go in your README and CV.

In [ ]:
import shutil
from google.colab import files

shutil.make_archive("/content/unet_results", "zip", "runs/unet")
files.download("landslide_unet.onnx")
files.download("/content/unet_results.zip")
print("ONNX model + metrics, history, curves and band stats downloaded.")